# Hurricane Harvey Flood Forecaster — Harris County, TX
### Run this notebook top to bottom in Google Colab

**Before you start:** Runtime menu -> Change runtime type -> select a GPU (T4/L4 are plenty for this model).

This notebook:
1. Clones the project repo
2. Installs dependencies
3. (Optionally) mounts Google Drive so downloaded data + trained checkpoints survive across Colab sessions
4. Downloads real USGS/NHC data for Hurricane Harvey (falls back to clearly-labeled synthetic data if those services are unreachable from this network)
5. Trains the model
6. Evaluates it and shows the hydrograph plots
7. Launches the interactive demo inline


In [ ]:
#@title 1. Clone the repo
# If you forked/cloned this elsewhere, change the URL below.
!git clone https://github.com/YOUR-USERNAME/harvey-flood-dl.git
%cd harvey-flood-dl


In [ ]:
#@title 2. Install dependencies
# Colab already ships torch/numpy/pandas/matplotlib/sklearn -- this mainly
# installs gradio and pyyaml, which are not preinstalled.
!pip install -q -r requirements.txt


## 3. (Optional but recommended) Mount Google Drive

Colab's local disk is wiped every time the runtime disconnects. Mounting
Drive lets `data/raw/` (downloaded USGS/HURDAT2 data) and `models/` (trained
checkpoint) persist across sessions, so you don't re-download or retrain
from scratch every time you come back to this.


In [ ]:
#@title Mount Drive and symlink data/models into it (skip this cell to stay local-only)
from google.colab import drive
drive.mount('/content/drive')

import os
drive_root = "/content/drive/MyDrive/harvey-flood-dl"
os.makedirs(f"{drive_root}/data/raw", exist_ok=True)
os.makedirs(f"{drive_root}/models", exist_ok=True)

# Point the repo's data/raw and models dirs at Drive so downloads + checkpoints persist
!rm -rf data/raw models
!ln -s "{drive_root}/data/raw" data/raw
!ln -s "{drive_root}/models" models
print("Linked data/raw and models to Google Drive.")


In [ ]:
#@title Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Build the dataset

This pulls, for each of the 4 zones:
- USGS NWIS gage height / discharge / precipitation (Aug 23 - Sep 2, 2017)
- NHC HURDAT2 best-track wind for Harvey

If either source is unreachable (rate-limited, network policy, service
hiccup), it automatically falls back to clearly-labeled **synthetic** data
so the rest of the notebook still runs. Check the printed warnings -- if you
see "SYNTHETIC fallback", the run used simulated data, not real Harvey
observations, for that zone.


In [ ]:
#@title Build dataset and preview it
from dataset import load_config, make_datasets
import matplotlib.pyplot as plt

cfg = load_config("config.yaml")
train_ds, val_ds, test_ds, normalizer, zone_frames = make_datasets(cfg)
print(f"Train windows: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

fig, axes = plt.subplots(len(zone_frames), 1, figsize=(10, 10), sharex=True)
for ax, (name, df) in zip(axes, zone_frames.items()):
    tag = " [SYNTHETIC]" if df["synthetic"].iloc[0] else " [REAL USGS/NHC DATA]"
    ax.plot(df.index, df["gage_height_ft"], color="black")
    ax.set_title(name + tag, fontsize=10)
    ax.set_ylabel("ft")
plt.tight_layout()
plt.show()


## 5. Train

In [ ]:
#@title Train the model
!python train.py --config config.yaml


In [ ]:
#@title View training curves
from IPython.display import Image
Image("outputs/training_curves.png")


## 6. Evaluate

In [ ]:
#@title Run evaluation (per-zone metrics + hydrograph plots)
!python evaluate.py --config config.yaml --checkpoint models/harvey_lstm.pt


In [ ]:
#@title View hydrograph plots
from IPython.display import Image, display
import glob
for path in sorted(glob.glob("outputs/hydrograph_*.png")):
    display(Image(path))


## 7. Demo

Pick a zone and a sustained rainfall/wind scenario, and see the model's
12-hour forecast. **Hold rainfall/wind fixed and switch the zone dropdown**
to see how the same storm input produces a different outcome depending on
where you are in the county.


In [ ]:
#@title Launch interactive demo (renders inline in Colab)
from demo import build_demo
demo = build_demo("models/harvey_lstm.pt")
demo.launch(share=True)  # share=True also gives you a public link you can send to others


## 8. Quick command-line scenario check (no UI)

Useful for scripting a handful of comparisons quickly.


In [ ]:
#@title CLI scenario comparison across all 4 zones
for zone in ["buffalo_bayou", "brays_bayou", "cypress_creek", "addicks_reservoir"]:
    print(f"=== {zone} ===")
    !python inference.py --zone {zone} --rainfall 3.0 --wind 50
    print()
